## End-to-End Demo

Single-track inference through the full Task 3 pipeline:

`audio -> segment graph -> GraphSAGE embedding` and `masked artist bio -> DistilBERT embedding`,
fused via the trained **cross-attention** model to predict contextual tags.

Uses the completed run at `results/task3/20260902-192256/` (all 4 ablation variants trained;
cross-attention test macro_f1=0.104, micro_f1=0.104, auc_pr=0.143 — see that run's `metrics.json`
for the full ablation comparison).

In [5]:
import json
import sys
from pathlib import Path

import torch
from transformers import AutoTokenizer

sys.path.insert(0, str(Path.cwd().parent))

from src.datasets import (
    build_task1_dataset,
    load_corrupted_track_ids,
    load_fma_metadata,
    load_fma_splits,
)
from src.fusion_model import FusionModel
from src.graph_builder import build_or_load_track_graph
from src.utils import load_config

ROOT = Path.cwd().parent
config = load_config(ROOT / "config.yaml")
device = "cuda" if torch.cuda.is_available() else "cpu"

RUN_DIR = ROOT / "results/task3/20260902-192256"
with (RUN_DIR / "metrics.json").open() as f:
    task3_metrics = json.load(f)
top_tags = task3_metrics["top_tags"]
thresholds = task3_metrics["ablations"]["cross_attention"]["per_tag_thresholds"]
num_labels = len(top_tags)

print(f"device={device}")
print(f"loaded run: {RUN_DIR.name}")
print(f"cross_attention test metrics: {task3_metrics['ablations']['cross_attention']['test']}")


device=cuda
loaded run: 20260902-192256
cross_attention test metrics: {'macro_f1': 0.10447818549656089, 'micro_f1': 0.10448979591836735, 'auc_pr': 0.14303209757266527, 'loss': 0.35763341800341586}


## Pick one test-set track

Uses the same leakage-safe Task 1 dataset construction as training: the input text is the
artist bio with every target-tag word (and the track's own genre) masked out with `[MASK]`.

In [6]:
tracks = load_fma_metadata(ROOT / "data/raw/fma_metadata", subset="medium")
corrupted_ids = load_corrupted_track_ids()
splits = load_fma_splits(tracks, exclude_track_ids=corrupted_ids)
task_df = build_task1_dataset(tracks, top_tags)
split_of = {tid: name for name, ids in splits.items() for tid in ids}
task_df = task_df.assign(split=task_df["track_id"].map(split_of))
test_df = task_df[task_df["split"] == "test"].reset_index(drop=True)

row = test_df[test_df["labels"].apply(sum) > 0].iloc[0]  # pick one with >=1 true tag, for a clearer demo
track_id = int(row["track_id"])
genre = tracks.loc[tracks["track_id"] == track_id, "genre_top"].item()
true_tags = [top_tags[i] for i, v in enumerate(row["labels"]) if v == 1]

print(f"track_id={track_id}  genre_top={genre}")
print(f"masked bio text: {row['text'][:300]}")
print(f"true (leakage-masked) contextual tags: {true_tags}")

track_id=13814  genre_top=Folk
masked bio text: Menhirs of Er Grah (Tom Carter) is a [MASK] band named after some prehistoric standing-stones in France. 
 Tom Carter lives in London, and has released a number of lo-fi/[MASK] albums under the anagram of March Rosetta.
true (leakage-masked) contextual tags: ['acoustic', 'clinical archives']


## Audio -> segment graph

In [7]:
audio_cfg = config["audio"]
graph_cfg = config["graph"]
in_dim = 2 * audio_cfg["n_mfcc"] + (24 if audio_cfg["use_chroma"] else 0)

graph = build_or_load_track_graph(
    track_id=track_id,
    audio_root=config["dataset"]["root"],
    cache_dir=ROOT / "data/processed/graph_cache",
    sample_rate=config["dataset"]["sample_rate"],
    segment_seconds=audio_cfg["segment_seconds"],
    n_mfcc=audio_cfg["n_mfcc"],
    use_chroma=audio_cfg["use_chroma"],
    similarity_threshold=graph_cfg["similarity_threshold"],
    bidirectional=graph_cfg["bidirectional_edges"],
    self_loops=graph_cfg["self_loops"],
)
print(f"segment graph for track {track_id}: {graph.x.shape[0]} nodes (5s segments), "
      f"{graph.edge_index.shape[1]} directed edges, node feature dim={graph.x.shape[1]}")

segment graph for track 13814: 6 nodes (5s segments), 10 directed edges, node feature dim=64


## Text -> BERT, fusion -> tag prediction

Loads the trained `cross_attention` fusion checkpoint and runs a single forward pass, then
applies the per-tag thresholds tuned on the validation set (same ones used at test time).

In [8]:
tokenizer = AutoTokenizer.from_pretrained(config["bert"]["model_name"])
encoded = tokenizer(
    row["text"], truncation=True, max_length=config["bert"]["max_length"],
    padding="max_length", return_tensors="pt",
)

model = FusionModel(
    graph_in_dim=in_dim,
    graph_hidden_dim=config["gnn"]["hidden_dim"],
    graph_num_layers=config["gnn"]["num_layers"],
    graph_dropout=config["gnn"]["dropout"],
    bert_model_name=config["bert"]["model_name"],
    bert_freeze_layers=config["bert"]["freeze_layers"],
    num_labels=num_labels,
    mode="cross_attention",
).to(device)
model.load_state_dict(torch.load(RUN_DIR / "cross_attention_best_model.pt", map_location=device))
model.eval()

batch_index = torch.zeros(graph.x.shape[0], dtype=torch.long)  # single graph -> single PyG "batch"
with torch.no_grad():
    logits = model(
        graph.x.to(device), graph.edge_index.to(device), batch_index.to(device),
        encoded["input_ids"].to(device), encoded["attention_mask"].to(device),
    )
    probs = torch.sigmoid(logits)[0].cpu().numpy()

predicted_tags = [top_tags[i] for i, p in enumerate(probs) if p >= thresholds[i]]
print(f"predicted contextual tags (cross-attention fusion): {predicted_tags}")
print(f"true contextual tags:                               {true_tags}")

Loading weights: 100%|██████████| 100/100 [00:00<00:00, 2441.46it/s]
[transformers] DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_transform.bias    | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


predicted contextual tags (cross-attention fusion): ['instrumental', 'acoustic', 'noise', 'clinical archives']
true contextual tags:                               ['acoustic', 'clinical archives']
